In [10]:
!pip install gymnasium
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print("OK")
def mc_control(epsilon=0.1, gamma=0.99, n_episodes=30000,
               eval_every=1000, eval_eps=300, max_steps=200):
    env = gym.make("FrozenLake-v1", is_slippery=True)
    nS, nA = env.observation_space.n, env.action_space.n
    Q         = np.zeros((nS, nA))
    Rsum      = np.zeros((nS, nA))
    Rcnt      = np.zeros((nS, nA))
    sr_hist, ar_hist, xpts = [], [], []
    for ep in range(1, n_episodes + 1):
        episode, s, _ = [], *env.reset()
        for _ in range(max_steps):
            p = np.ones(nA) * epsilon / nA
            p[np.argmax(Q[s])] += 1 - epsilon
            a = np.random.choice(nA, p=p)
            ns, r, term, trunc, _ = env.step(a)
            episode.append((s, a, r))
            s = ns
            if term or trunc:
                break
        G, visited = 0.0, set()
        for s, a, r in reversed(episode):
            G = gamma * G + r
            if (s, a) not in visited:
                visited.add((s, a))
                Rsum[s, a] += G
                Rcnt[s, a] += 1
                Q[s, a] = Rsum[s, a] / Rcnt[s, a]
        if ep % eval_every == 0:
            g = np.argmax(Q, axis=1)
            wins, rets = 0, []
            for _ in range(eval_eps):
                s2, _ = env.reset()
                tot, done, st = 0, False, 0
                while not done and st < max_steps:
                    s2, rw, t, tr, _ = env.step(g[s2])
                    tot += rw; st += 1; done = t or tr
                rets.append(tot)
                wins += (tot > 0)
            sr_hist.append(wins / eval_eps)
            ar_hist.append(np.mean(rets))
            xpts.append(ep)
            print(f"  ε={epsilon} | ep {ep:5d} | SR={wins/eval_eps:.3f} | AR={np.mean(rets):.3f}")
    g = np.argmax(Q, axis=1)
    wins, rets, steps = 0, [], []
    for _ in range(1000):
        s2, _ = env.reset()
        tot, done, st = 0, False, 0
        while not done and st < max_steps:
            s2, rw, t, tr, _ = env.step(g[s2])
            tot += rw; st += 1; done = t or tr
        rets.append(tot)
        if tot > 0:
            wins += 1; steps.append(st)
    env.close()
    return dict(sr=wins/1000, ar=np.mean(rets),
                steps=np.mean(steps) if steps else float('nan'),
                sr_hist=sr_hist, ar_hist=ar_hist, xpts=xpts)

print("mc_control defined")
print("Running ε = 0.1  (takes ~3–5 min)...")
r01 = mc_control(epsilon=0.1)
print("Running ε = 0.3  (takes ~3–5 min)...")
r03 = mc_control(epsilon=0.3)
print(f"ε=0.1 → Success Rate: {r01['sr']:.3f} | Avg Return: {r01['ar']:.3f} | Avg Steps: {r01['steps']:.1f}")
print(f"ε=0.3 → Success Rate: {r03['sr']:.3f} | Avg Return: {r03['ar']:.3f} | Avg Steps: {r03['steps']:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Success Rate
axes[0].plot(r01["xpts"], r01["sr_hist"], label="ε=0.1", linewidth=2)
axes[0].plot(r03["xpts"], r03["sr_hist"], label="ε=0.3", linewidth=2, linestyle="--")
axes[0].set_xlabel("Training Episodes", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Success Rate",       fontsize=12, fontweight="bold")
axes[0].set_title("Success Rate vs Training Episodes", fontsize=13, fontweight="bold")
axes[0].set_ylim(0, 1); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Average Return
axes[1].plot(r01["xpts"], r01["ar_hist"], label="ε=0.1", linewidth=2)
axes[1].plot(r03["xpts"], r03["ar_hist"], label="ε=0.3", linewidth=2, linestyle="--")
axes[1].set_xlabel("Training Episodes", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Average Return",    fontsize=12, fontweight="bold")
axes[1].set_title("Average Return vs Training Episodes", fontsize=13, fontweight="bold")
axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("success_rate_vs_episodes.png",   dpi=300, bbox_inches="tight")
plt.savefig("average_return_vs_episodes.png", dpi=300, bbox_inches="tight")
plt.show()
from google.colab import files
files.download("success_rate_vs_episodes.png")
files.download("average_return_vs_episodes.png")
print(" Downloads triggered")

OK
mc_control defined
Running ε = 0.1  (takes ~3–5 min)...
  ε=0.1 | ep  1000 | SR=0.000 | AR=0.000
  ε=0.1 | ep  2000 | SR=0.000 | AR=0.000
  ε=0.1 | ep  3000 | SR=0.000 | AR=0.000
  ε=0.1 | ep  4000 | SR=0.000 | AR=0.000
  ε=0.1 | ep  5000 | SR=0.013 | AR=0.013
  ε=0.1 | ep  6000 | SR=0.013 | AR=0.013
  ε=0.1 | ep  7000 | SR=0.020 | AR=0.020
  ε=0.1 | ep  8000 | SR=0.027 | AR=0.027
  ε=0.1 | ep  9000 | SR=0.030 | AR=0.030
  ε=0.1 | ep 10000 | SR=0.027 | AR=0.027
  ε=0.1 | ep 11000 | SR=0.073 | AR=0.073
  ε=0.1 | ep 12000 | SR=0.067 | AR=0.067
  ε=0.1 | ep 13000 | SR=0.077 | AR=0.077
  ε=0.1 | ep 14000 | SR=0.087 | AR=0.087
  ε=0.1 | ep 15000 | SR=0.190 | AR=0.190
  ε=0.1 | ep 16000 | SR=0.140 | AR=0.140
  ε=0.1 | ep 17000 | SR=0.143 | AR=0.143
  ε=0.1 | ep 18000 | SR=0.147 | AR=0.147
  ε=0.1 | ep 19000 | SR=0.170 | AR=0.170
  ε=0.1 | ep 20000 | SR=0.217 | AR=0.217
  ε=0.1 | ep 21000 | SR=0.150 | AR=0.150
  ε=0.1 | ep 22000 | SR=0.193 | AR=0.193
  ε=0.1 | ep 23000 | SR=0.170 | AR=0.17

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Downloads triggered
